# SIFT-Based Image Matching

**Scale-Invariant Feature Transform (SIFT)** detects and describes local image features that are invariant to scale, rotation, and partially invariant to illumination and viewpoint changes.

## Pipeline
1. **Keypoint detection** — Find scale-space extrema (DoG pyramid).
2. **Descriptor computation** — 128-dimensional gradient histogram per keypoint.
3. **Matching** — FLANN-based kNN matching + Lowe's ratio test to keep reliable matches.
4. **Homography estimation** — RANSAC robustly estimates the geometric transform between the two views.
5. **Visualisation** — Draw matched keypoints and the projected object boundary.

Images are downloaded automatically — no local files required.

---
## 1 · Imports & Helpers

In [ ]:
import os
import urllib.request

import cv2
import numpy as np
import matplotlib.pyplot as plt

print(f"OpenCV  : {cv2.__version__}")
print(f"NumPy   : {np.__version__}")


def fetch(url: str, filename: str) -> np.ndarray:
    """Download `url` if not cached, return as BGR ndarray."""
    if not os.path.exists(filename):
        print(f"Downloading {filename} …")
        urllib.request.urlretrieve(url, filename)
    img = cv2.imread(filename)
    if img is None:
        raise RuntimeError(f"Could not decode: {filename}")
    return img


def show(images, titles, figsize=(16, 5)):
    """Display one or more BGR/gray images side-by-side."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        disp = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img
        ax.imshow(disp, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


print("Helpers loaded ✓")

---
## 2 · Load Images

We use the classic **graf** image pair from the Oxford affine benchmark — two photographs of the same graffiti wall taken from different viewpoints. This pair is commonly used to evaluate feature matchers.

In [ ]:
BASE = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/"

img1 = fetch(BASE + "graf1.png", "sift_img1.png")  # reference
img2 = fetch(BASE + "graf3.png", "sift_img2.png")  # rotated / zoomed

gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

print(f"Image 1 shape : {img1.shape}")
print(f"Image 2 shape : {img2.shape}")

show([img1, img2], ["Image 1 — graf1 (reference)", "Image 2 — graf3 (different viewpoint)"])

---
## 3 · SIFT Keypoint Detection & Descriptor Computation

SIFT builds a **Difference-of-Gaussian** (DoG) scale-space pyramid and finds stable extrema as keypoints. Each keypoint is then described by a 128-D histogram of gradient orientations drawn from a 16×16 neighbourhood, making the descriptor robust to rotation and scale.

In [ ]:
sift = cv2.SIFT_create()

kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

print(f"Image 1 keypoints : {len(kp1):5d}")
print(f"Image 2 keypoints : {len(kp2):5d}")
print(f"Descriptor shape  : {des1.shape}  (n × 128)")

# Draw keypoints (size and orientation)
kp_img1 = cv2.drawKeypoints(
    gray1, kp1, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)
kp_img2 = cv2.drawKeypoints(
    gray2, kp2, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

show([kp_img1, kp_img2],
     [f"SIFT keypoints — Image 1 ({len(kp1)})",
      f"SIFT keypoints — Image 2 ({len(kp2)})"])

---
## 4 · Descriptor Matching — FLANN + Lowe's Ratio Test

**FLANN** (Fast Library for Approximate Nearest Neighbours) efficiently performs kNN search in high-dimensional descriptor space.  
**Lowe's ratio test** keeps only matches where the best match distance is significantly smaller than the second-best:

$$
\frac{d(m_1)}{d(m_2)} < \text{ratio\_thresh}
$$

A lower ratio threshold yields fewer but more reliable matches.

In [ ]:
RATIO_THRESH = 0.75

# FLANN parameters for SIFT (float descriptors → KD-tree)
index_params  = dict(algorithm=1, trees=5)   # FLANN_INDEX_KDTREE = 1
search_params = dict(checks=50)

flann   = cv2.FlannBasedMatcher(index_params, search_params)
raw     = flann.knnMatch(des1, des2, k=2)
good    = [m for m, n in raw if m.distance < RATIO_THRESH * n.distance]

print(f"Raw candidate pairs   : {len(raw)}")
print(f"After Lowe ratio test : {len(good)}  (ratio threshold = {RATIO_THRESH})")

---
## 5 · Visualise Matched Keypoints

In [ ]:
# Show the top 60 matches for clarity
DISPLAY_N = 60
match_img = cv2.drawMatches(
    img1, kp1, img2, kp2,
    good[:DISPLAY_N], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(18, 6))
plt.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
plt.title(f"SIFT Feature Matching — top {DISPLAY_N} of {len(good)} good matches", fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

---
## 6 · Homography Estimation with RANSAC

**Homography** maps every point in Image 1 to its corresponding location in Image 2 under a projective transformation (8 DOF).  
**RANSAC** (Random Sample Consensus) robustly fits the model while automatically rejecting outlier matches.

The detected **inliers** (matches consistent with the estimated homography) are drawn in green; **outliers** are shown in red.

In [ ]:
MIN_MATCH = 10

if len(good) < MIN_MATCH:
    print(f"Only {len(good)} matches — need at least {MIN_MATCH}. Reduce RATIO_THRESH.")
else:
    # Extract point coordinates
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    # Estimate homography
    H, mask = cv2.findHomography(
        src_pts, dst_pts,
        cv2.RANSAC, ransacReprojThreshold=5.0
    )
    inlier_count = int(mask.sum()) if mask is not None else 0

    print(f"Total good matches : {len(good)}")
    print(f"RANSAC inliers     : {inlier_count}")
    print(f"Outliers rejected  : {len(good) - inlier_count}")
    print("\nHomography matrix (3×3):")
    print(np.round(H, 4))

    # Draw inlier / outlier matches
    inlier_mask_list = mask.ravel().tolist() if mask is not None else None
    vis = cv2.drawMatches(
        img1, kp1, img2, kp2, good, None,
        matchesMask=inlier_mask_list,
        matchColor=(0, 220, 0),
        singlePointColor=(180, 0, 0),
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )
    plt.figure(figsize=(18, 6))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"RANSAC — {inlier_count} inliers (green) / {len(good)-inlier_count} outliers (red)",
              fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print("\n✓ Homography estimated successfully.")

---
## 7 · Project Image 1 Boundary into Image 2

Using the estimated homography, we warp the four corners of Image 1 into Image 2's coordinate frame and draw the bounding polygon, confirming that the spatial mapping is geometrically correct.

In [ ]:
h1, w1 = img1.shape[:2]
corners = np.float32([[0, 0], [w1, 0], [w1, h1], [0, h1]]).reshape(-1, 1, 2)
corners_warped = cv2.perspectiveTransform(corners, H)

# Draw boundary polygon on a copy of img2
scene = img2.copy()
cv2.polylines(scene, [np.int32(corners_warped)],
              isClosed=True, color=(0, 255, 0), thickness=3, lineType=cv2.LINE_AA)

for pt, lbl in zip(corners_warped.reshape(-1, 2), ["TL", "TR", "BR", "BL"]):
    cv2.putText(scene, lbl, (int(pt[0]) + 6, int(pt[1]) - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)

show([img1, scene],
     ["Image 1 (source)", "Image 2 — projected boundary of Image 1 (green)"],
     figsize=(14, 6))

print(f"Projected corner coordinates (px): {np.int32(corners_warped).reshape(-1,2).tolist()}")
print("\n✓ SIFT matching complete.")